In [0]:
# 1. Read the bronze tables
df_leads = spark.table("my_project_db.bronze_leads")
df_users = spark.table("my_project_db.users")
df_source = spark.table("my_project_db.lead_source_lookup")
df_status = spark.table("my_project_db.lead_status_lookup")

# 2. Join and Denormalize
# Using specific column names: ls_name for source, lst_name for status
df_silver_leads = df_leads.alias("l") \
    .join(df_users.alias("u"), df_leads.user_id == df_users.user_id, "left") \
    .join(df_source.alias("s"), df_leads.source_id == df_source.source_id, "left") \
    .join(df_status.alias("st"), df_leads.status_id == df_status.status_id, "left") \
    .select(
        "l.lead_id",
        "l.lead_date",
        "l.invt_id",
        "l.quantity",
        "l.customer_type",
        "u.username",
        "u.region",
        "s.ls_name",       
        "st.lst_name"
    ) \
    .withColumnRenamed("ls_name", "source_name") \
    .withColumnRenamed("lst_name", "status_name")

# 3. Save to Silver
df_silver_leads.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("my_project_db.silver_leads")

print("silver_leads table created successfully with updated naming convention!")
df_silver_leads.limit(5).display()

In [0]:
from pyspark.sql.functions import to_date, col

# Load and clean
df_silver_leads = spark.table("my_project_db.silver_leads") \
    .withColumn("lead_date", to_date(col("lead_date")))

# Use overwriteSchema to explicitly allow changing the column type
df_silver_leads.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("my_project_db.silver_leads")

print("Schema updated successfully!")

In [0]:
df_silver_leads.limit(5).display()

In [0]:
# Check schema of bronze_inventory
df_inv = spark.table("my_project_db.bronze_inventory")
df_inv.printSchema()
df_inv.limit(5).display()

In [0]:
from pyspark.sql.functions import col

# 1. Read the bronze inventory and the inventory lookup table
df_inv = spark.table("my_project_db.bronze_inventory")
df_inv_status = spark.table("my_project_db.status_lookup") 

# 2. Join and Denormalize
df_silver_inventory = df_inv.alias("inv") \
    .join(df_inv_status.alias("st"), df_inv.status_id == df_inv_status.status_id, "left") \
    .select(
        "inv.invt_id",
        "inv.product_sku",
        "inv.purchase_cost",
        "inv.quantity",
        to_date(col("inv.warranty_date")).alias("warranty_date"), # Cleaned date format
        "inv.division",
        "st.name"
    ) \
    .withColumnRenamed("name", "status_name")

# 3. Save as Silver with schema overwrite
df_silver_inventory.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("my_project_db.silver_inventory")

print("silver_inventory table updated with clean warranty_date!")
df_silver_inventory.limit(5).display()


In [0]:
from pyspark.sql.functions import col, to_date

# 1. Read the tables
df_orders = spark.table("my_project_db.bronze_orders")
df_status = spark.table("my_project_db.order_status_lookup")
df_reasons = spark.table("my_project_db.order_reasons_lookup")

# 2. Join, Denormalize, and Clean Dates
# We join 'status' twice: once for initial and once for final status
df_silver_orders = df_orders.alias("o") \
    .join(df_status.alias("init_st"), col("o.initial_status_id") == col("init_st.status_id"), "left") \
    .join(df_status.alias("final_st"), col("o.final_status_id") == col("final_st.status_id"), "left") \
    .join(df_reasons.alias("r"), col("o.reason_id") == col("r.reason_id"), "left") \
    .select(
        "o.order_id",
        "o.lead_id",
        to_date(col("o.order_in_date")).alias("order_in_date"),
        to_date(col("o.order_out_date")).alias("order_out_date"),
        "o.order_value",
        col("init_st.os_name").alias("initial_status_name"),
        col("final_st.os_name").alias("final_status_name"),
        col("r.rname").alias("reason_name")
    )

# Save to Silver
df_silver_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("my_project_db.silver_orders")
    
print("silver_orders table created/updated successfully!")
df_silver_orders.limit(5).display()